# MediSphere Cognitive Twin — Custom TensorFlow Federated Learning Pipeline
### Privacy-Preserving Collaborative Training across Simulated Hospitals for CVD and Diabetes Risk Prediction

---

## Executive Overview
In modern digital healthcare, machine learning models trained on single-hospital datasets suffer from **geographic and demographic sampling bias**. However, aggregating patient Electronic Health Records (EHR) into a central repository violates data privacy laws (**HIPAA, GDPR, DISHA**) and exposes sensitive patient records to security breaches.

**Federated Learning (FL)** solves this dilemma:
1. Patient data **never leaves the hospital's local boundary**.
2. Each hospital trains a local model on its internal patient cohort.
3. Only **model parameters (weights and biases)** are transmitted to a central coordinator.
4. The central coordinator performs **Federated Averaging (FedAvg)** to update a shared **Global Model**, which is then redistributed to all hospitals.

```
                  +-----------------------------------+
                  | Central Federated Coordinator     |
                  | Global Aggregation Engine (FedAvg)|
                  +-----------------+-----------------+
                                    ^
                   Global Weights   |  Local Parameter
                   Broadcast        |  Updates Only
                                    v
     +------------------------------+------------------------------+
     |                              |                              |
     v                              v                              v
+----+------------------+  +--------+------------------+  +-------+------------------+
| Hospital 1 (Node)     |  | Hospital 2 (Node)         |  | Hospital 3 (Node)        |
| Regional Medical Ctr  |  | Urban University Hospital |  | Community Clinic         |
| Private Data: 40%     |  | Private Data: 35%         |  | Private Data: 25%        |
| Local Keras Model     |  | Local Keras Model         |  | Local Keras Model        |
+-----------------------+  +---------------------------+  +--------------------------+
```

---
## Implementation Engine: Custom TensorFlow FedAvg
This notebook implements **Custom TensorFlow Federated Averaging** built natively on **TensorFlow 2.20.0 / Keras** and **NumPy**.
- **No external TFF framework dependency** required.
- Standard `tf.keras` neural networks for hospital node models.
- Parameter-level weighted aggregation: $W_{global} = \sum_{k=1}^K \frac{n_k}{N} W_k$.

---
## Pipeline Roadmap
1. **Environment Setup**: Verify Python & TensorFlow 2.20.0 setup.
2. **Dataset Loading**: Load Framingham CVD and Pima Indians Diabetes datasets.
3. **Preprocessing**: Clean, impute, encode, and scale features (zero data leakage).
4. **Hospital Partitioning**: Divide training data into 3 isolated simulated hospital nodes (40%, 35%, 25%).
5. **Model Architecture**: Identical Keras MLP for all hospital nodes.
6. **Custom FedAvg Engine**: Reusable functions for local training, weight aggregation, and global updates.
7. **Federated Training**: Multi-round collaborative learning across 10 federated rounds.
8. **Global Model Evaluation**: Evaluate aggregated global models on unseen 20% holdout set.
9. **Convergence Visualizations**: Plot loss and accuracy trajectories across rounds.
10. **Model Export**: Save models + metadata for local Flask ML API integration.


## 1. Environment Setup & Runtime Verification

We use native TensorFlow 2.20.0 and standard Python ML libraries (NumPy, pandas, scikit-learn, matplotlib). No external federated framework packages are needed.


In [ ]:
import os
import sys
import json
import datetime
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Suppress verbose TensorFlow C++ logs
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

print("=" * 65)
print("   MEDISPHERE FEDERATED LEARNING -- ENVIRONMENT DIAGNOSTICS   ")
print("=" * 65)
print(f"Python Version:             {sys.version.split()[0]}")
print(f"TensorFlow Version:         {tf.__version__}")
print("Federated Learning Backend: Custom TensorFlow FedAvg")
print("=" * 65)


## 2. Dataset Loading & Configuration

We use two datasets from the MediSphere Cognitive Twin repository:
1. **CVD Risk**: Framingham Heart Study — `framingham_cleaned.csv` (preferred) or `framingham.csv`
2. **Diabetes Risk**: Pima Indians Diabetes Database — `diabetes.csv`

The loading function automatically checks both `data/` subfolder and root folder.


In [ ]:
# ============================================================
# CONFIGURATION: DATASET FILE PATHS
# Checks both current folder and data/ subfolder
# ============================================================
CVD_CANDIDATES = [
    "data/framingham_cleaned.csv", "framingham_cleaned.csv",
    "data/framingham.csv", "framingham.csv"
]
DIABETES_CANDIDATES = [
    "data/diabetes.csv", "diabetes.csv"
]

cvd_data_path = None
for p in CVD_CANDIDATES:
    if os.path.exists(p):
        cvd_data_path = p
        break

diab_data_path = None
for p in DIABETES_CANDIDATES:
    if os.path.exists(p):
        diab_data_path = p
        break

# Prompt upload if missing in Colab/Notebook environment
missing = []
if not cvd_data_path:
    missing.append("CVD dataset -> framingham_cleaned.csv or framingham.csv")
if not diab_data_path:
    missing.append("Diabetes dataset -> diabetes.csv")

if missing:
    print("The following dataset files were NOT found in the working directory:")
    for m in missing:
        print(f"  * {m}")
    try:
        from google.colab import files
        print("\nLaunching Colab file upload dialog...")
        files.upload()
        # Re-check after upload
        for p in CVD_CANDIDATES:
            if os.path.exists(p):
                cvd_data_path = p
                break
        for p in DIABETES_CANDIDATES:
            if os.path.exists(p):
                diab_data_path = p
                break
    except Exception:
        print("\nPlease place dataset CSV files in data/ or working directory.")

if not cvd_data_path:
    raise FileNotFoundError("CVD dataset not found. Please provide 'framingham_cleaned.csv' or 'framingham.csv'.")

if not diab_data_path:
    raise FileNotFoundError("Diabetes dataset not found. Please provide 'diabetes.csv'.")

print(f"CVD dataset path:      {cvd_data_path}")
print(f"Diabetes dataset path: {diab_data_path}")


## 3. Data Preprocessing & Validation

Both datasets are preprocessed using strict train/test isolation to prevent data leakage:

**CVD (Framingham) Pipeline:**
- Remove `id` column if present.
- Encode categorical columns: `sex` (F→0, M→1), `is_smoking`/`currentSmoker` (NO→0, YES→1).
- Impute missing numerical values using **training-set medians only**.
- Target: `TenYearCHD` — 10-year coronary heart disease risk (0/1).

**Diabetes (Pima Indians) Pipeline:**
- Replace physiologically impossible zeros in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI` with `NaN`, then impute with **training-set medians**.
- Target: `Outcome` — diabetes diagnosis (0/1).

**Leakage Prevention:** `StandardScaler` is fitted *only* on training data (80%), then applied to both train and test sets. The holdout test set (20%) is kept completely unseen until final evaluation.


In [ ]:
# ============================================================
# 3.1 PREPROCESS CVD (FRAMINGHAM) DATASET
# ============================================================
print("=" * 65)
print("PREPROCESSING CVD DATASET")
print("=" * 65)

df_cvd = pd.read_csv(cvd_data_path)
print(f"Loaded shape: {df_cvd.shape}")

# Drop ID column if present
if "id" in df_cvd.columns:
    df_cvd = df_cvd.drop(columns=["id"])

# Encode categorical columns if they are still strings
if "sex" in df_cvd.columns and df_cvd["sex"].dtype == object:
    df_cvd["sex"] = df_cvd["sex"].map({"F": 0, "M": 1})

smoking_col = "is_smoking" if "is_smoking" in df_cvd.columns else "currentSmoker"
if smoking_col in df_cvd.columns and df_cvd[smoking_col].dtype == object:
    df_cvd[smoking_col] = df_cvd[smoking_col].map({"NO": 0, "YES": 1, "no": 0, "yes": 1})

TARGET_CVD = "TenYearCHD"
X_cvd = df_cvd.drop(columns=[TARGET_CVD]).copy()
y_cvd = df_cvd[TARGET_CVD].astype(int).copy()
cvd_feature_names = list(X_cvd.columns)

print(f"Feature columns ({len(cvd_feature_names)}): {cvd_feature_names}")
print(f"Target column: {TARGET_CVD}")
print(f"Class distribution: {dict(y_cvd.value_counts())}")

# Train/Test split (80/20, stratified, random_state=42)
X_cvd_tr, X_cvd_te, y_cvd_tr, y_cvd_te = train_test_split(
    X_cvd, y_cvd, test_size=0.20, random_state=42, stratify=y_cvd
)

# Impute missing values using TRAINING medians only
for col in X_cvd_tr.columns:
    med = X_cvd_tr[col].median()
    X_cvd_tr[col] = X_cvd_tr[col].fillna(med)
    X_cvd_te[col] = X_cvd_te[col].fillna(med)

# Scale features using TRAINING statistics only
scaler_cvd = StandardScaler()
X_cvd_train = scaler_cvd.fit_transform(X_cvd_tr).astype(np.float32)
X_cvd_test  = scaler_cvd.transform(X_cvd_te).astype(np.float32)
y_cvd_train = y_cvd_tr.to_numpy().astype(np.float32)
y_cvd_test  = y_cvd_te.to_numpy().astype(np.float32)

print(f"\nCVD Train: {X_cvd_train.shape[0]} patients | Test: {X_cvd_test.shape[0]} patients")
print(f"Train class balance: {int(y_cvd_train.sum())} positive / {int(len(y_cvd_train)-y_cvd_train.sum())} negative")

# ============================================================
# 3.2 PREPROCESS DIABETES (PIMA INDIANS) DATASET
# ============================================================
print("\n" + "=" * 65)
print("PREPROCESSING DIABETES DATASET")
print("=" * 65)

df_diab = pd.read_csv(diab_data_path)
print(f"Loaded shape: {df_diab.shape}")

TARGET_DIAB = "Outcome"
X_diab = df_diab.drop(columns=[TARGET_DIAB]).copy()
y_diab = df_diab[TARGET_DIAB].astype(int).copy()
diab_feature_names = list(X_diab.columns)

print(f"Feature columns ({len(diab_feature_names)}): {diab_feature_names}")
print(f"Class distribution: {dict(y_diab.value_counts())}")

# Train/Test split (80/20, stratified, random_state=42)
X_diab_tr, X_diab_te, y_diab_tr, y_diab_te = train_test_split(
    X_diab, y_diab, test_size=0.20, random_state=42, stratify=y_diab
)

# Replace physiologically impossible zeros with NaN, then impute with training medians
invalid_zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in invalid_zero_cols:
    if col in X_diab_tr.columns:
        X_diab_tr[col] = X_diab_tr[col].replace(0, np.nan)
        X_diab_te[col] = X_diab_te[col].replace(0, np.nan)
        med = X_diab_tr[col].median()
        X_diab_tr[col] = X_diab_tr[col].fillna(med)
        X_diab_te[col] = X_diab_te[col].fillna(med)

# Scale features using TRAINING statistics only
scaler_diab = StandardScaler()
X_diab_train = scaler_diab.fit_transform(X_diab_tr).astype(np.float32)
X_diab_test  = scaler_diab.transform(X_diab_te).astype(np.float32)
y_diab_train = y_diab_tr.to_numpy().astype(np.float32)
y_diab_test  = y_diab_te.to_numpy().astype(np.float32)

print(f"\nDiabetes Train: {X_diab_train.shape[0]} patients | Test: {X_diab_test.shape[0]} patients")
print(f"Train class balance: {int(y_diab_train.sum())} positive / {int(len(y_diab_train)-y_diab_train.sum())} negative")
print("\nPreprocessing complete.")


## 4. Simulated Hospital Partitioning

### Educational Simulation Disclaimer
> **NOTE**: The three "hospitals" below are created by partitioning a public research dataset to simulate a multi-center healthcare environment. No actual hospital EHR databases or live clinical networks are connected.

Partition Distribution:
- **Hospital 1 — Regional Medical Center**: 40% of training data
- **Hospital 2 — Urban University Hospital**: 35% of training data
- **Hospital 3 — Community Health Clinic**: 25% of training data

Random seed `random_state = 42` ensures exact reproducibility across runs.


In [ ]:
HOSPITAL_NAMES = [
    "Hospital 1 (Regional Medical Center)",
    "Hospital 2 (Urban University Hospital)",
    "Hospital 3 (Community Health Clinic)"
]
HOSPITAL_SHARES = [0.40, 0.35, 0.25]
RANDOM_SEED = 42

def partition_into_hospitals(X_data, y_data, shares=HOSPITAL_SHARES, seed=RANDOM_SEED):
    """
    Partitions training data into isolated simulated hospital cohorts.
    No hospital partition overlaps with another (strict privacy boundary simulation).
    """
    assert abs(sum(shares) - 1.0) < 1e-6, "Hospital shares must sum to 1.0"
    np.random.seed(seed)
    idx = np.random.permutation(len(X_data))
    
    n1 = int(shares[0] * len(X_data))
    n2 = int(shares[1] * len(X_data))
    
    return [
        {"name": HOSPITAL_NAMES[0], "X": X_data[idx[:n1]],        "y": y_data[idx[:n1]]},
        {"name": HOSPITAL_NAMES[1], "X": X_data[idx[n1:n1+n2]],   "y": y_data[idx[n1:n1+n2]]},
        {"name": HOSPITAL_NAMES[2], "X": X_data[idx[n1+n2:]],     "y": y_data[idx[n1+n2:]]},
    ]

cvd_hospitals  = partition_into_hospitals(X_cvd_train, y_cvd_train)
diab_hospitals = partition_into_hospitals(X_diab_train, y_diab_train)

for label, hospitals, total in [("CVD", cvd_hospitals, len(X_cvd_train)),
                                  ("DIABETES", diab_hospitals, len(X_diab_train))]:
    print(f"\n{'='*65}")
    print(f"  {label} -- 3 SIMULATED HOSPITAL PARTITIONS")
    print(f"{'='*65}")
    for h in hospitals:
        pos = int(h['y'].sum())
        print(f"  {h['name']}")
        print(f"    Patients : {len(h['X'])}  ({len(h['X'])/total*100:.1f}% of training set)")
        print(f"    Positive : {pos}  |  Negative : {len(h['y'])-pos}  ({pos/len(h['y'])*100:.1f}% positive rate)")


## 5. Neural Network Architecture

Every simulated hospital node uses an **identical Keras MLP architecture** so that weight tensors are structurally compatible for aggregation.

Architecture:
```
Input(num_features)
  |
Dense(32, activation="relu")
  |
Dropout(0.20)
  |
Dense(16, activation="relu")
  |
Dropout(0.10)
  |
Dense(1, activation="sigmoid")
```
- **Optimizer**: `tf.keras.optimizers.Adam(learning_rate=0.005)`
- **Loss Function**: `tf.keras.losses.BinaryCrossentropy()`
- **Metrics**: `BinaryAccuracy(name="accuracy")`, `AUC(name="auc")`


In [ ]:
def create_model(input_dim, name="risk_model"):
    """
    Constructs a tabular binary classification MLP model in Keras.
    Returns a fresh compiled Keras Sequential model.
    """
    model = keras.Sequential([
        layers.Input(shape=(input_dim,), name="clinical_features"),
        layers.Dense(32, activation="relu", name="dense_1"),
        layers.Dropout(0.20, name="dropout_1"),
        layers.Dense(16, activation="relu", name="dense_2"),
        layers.Dropout(0.10, name="dropout_2"),
        layers.Dense(1, activation="sigmoid", name="risk_output")
    ], name=name)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.005),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

# Print architectural check
sample_model = create_model(len(cvd_feature_names), "cvd_architecture_check")
sample_model.summary()


## 6. Custom TensorFlow Federated Averaging (FedAvg) Core Functions

Below are the 3 modular functions implementing Federated Averaging:

1. `train_local_model(global_weights, X_local, y_local, local_epochs, batch_size)`:
   - Instantiates a fresh local Keras model.
   - Loads current global weights via `.set_weights()`.
   - Trains locally on hospital data for `local_epochs`.
   - Returns updated local weight tensors via `.get_weights()`.

2. `fedavg(local_weights_list, local_sample_counts)`:
   - Computes hospital sample weight ratio $w_k = n_k / N$.
   - Computes weighted average for each neural network weight tensor:
     $$W_{\text{global}}[i] = \sum_{k=1}^K w_k \cdot W_{\text{local}}^k[i]$$
   - Returns aggregated weight list.

3. `run_federated_training(task_name, input_dim, hospitals_data, X_test, y_test, num_rounds, local_epochs, batch_size)`:
   - Orchestrates multi-round federated training, global model updates, and per-round test evaluation.


In [ ]:
def train_local_model(global_weights, X_local, y_local, local_epochs=3, batch_size=32):
    """
    Simulates local hospital training:
    1. Creates a local model and initializes it with current global weights.
    2. Trains locally on the hospital's private dataset.
    3. Returns local updated weight tensors and training history.
    """
    input_dim = X_local.shape[1]
    local_model = create_model(input_dim)
    local_model.set_weights(global_weights)

    history = local_model.fit(
        X_local,
        y_local,
        epochs=local_epochs,
        batch_size=batch_size,
        verbose=0
    )
    
    local_weights = local_model.get_weights()
    final_loss = history.history["loss"][-1]
    final_acc  = history.history["accuracy"][-1]

    return local_weights, final_loss, final_acc


def fedavg(local_weights_list, local_sample_counts):
    """
    Performs sample-weighted Federated Averaging over parameter tensors.
    
    Formula:
      W_global = sum_k (n_k / N) * W_k
      where n_k is sample count for hospital k, and N is total samples.
    """
    N = sum(local_sample_counts)
    weights_per_hospital = [n / N for n in local_sample_counts]

    num_tensors = len(local_weights_list[0])
    aggregated_weights = []

    for tensor_idx in range(num_tensors):
        # Element-wise weighted average across all hospital parameter matrices
        weighted_sum = sum(
            weights_per_hospital[k] * local_weights_list[k][tensor_idx]
            for k in range(len(local_weights_list))
        )
        aggregated_weights.append(weighted_sum)

    return aggregated_weights


def run_federated_training(
    task_name,
    input_dim,
    hospitals_data,
    X_test,
    y_test,
    num_rounds=10,
    local_epochs=3,
    batch_size=32
):
    """
    Main Federated Averaging training loop.
    Coordinates weight broadcasting, local SGD, FedAvg parameter aggregation,
    and holdout test set monitoring across federated rounds.
    """
    print(f"\n{'='*70}")
    print(f"  FEDERATED LEARNING: {task_name.upper()}   |   3 SIMULATED HOSPITALS")
    print(f"  Backend: Custom TensorFlow FedAvg   |   Rounds: {num_rounds}   |   Local Epochs: {local_epochs}")
    print(f"{'='*70}\n")

    # 1. Initialize Global Model
    global_model = create_model(input_dim, name=f"global_{task_name}")
    current_global_weights = global_model.get_weights()

    sample_counts = [len(h["X"]) for h in hospitals_data]
    total_samples = sum(sample_counts)
    history_records = []

    for round_num in range(1, num_rounds + 1):
        print(f"--- Federated Round {round_num}/{num_rounds} ---")

        local_weights_list = []
        local_losses = []
        local_accs = []

        # 2. Local Training at each Simulated Hospital Node
        for k, hospital in enumerate(hospitals_data):
            l_weights, l_loss, l_acc = train_local_model(
                global_weights=current_global_weights,
                X_local=hospital["X"],
                y_local=hospital["y"],
                local_epochs=local_epochs,
                batch_size=batch_size
            )
            local_weights_list.append(l_weights)
            local_losses.append(l_loss)
            local_accs.append(l_acc)

            print(f"  {hospital['name']}: {len(hospital['X'])} patients | Local Loss: {l_loss:.4f} | Local Acc: {l_acc:.4f}")

        # 3. Perform Parameter Weighted FedAvg Aggregation
        current_global_weights = fedavg(local_weights_list, sample_counts)
        global_model.set_weights(current_global_weights)

        # 4. Compute Weighted Local Train Metrics & Evaluate Global Model on Centralized Test Set
        avg_train_loss = sum((sample_counts[k] / total_samples) * local_losses[k] for k in range(len(hospitals_data)))
        avg_train_acc  = sum((sample_counts[k] / total_samples) * local_accs[k]   for k in range(len(hospitals_data)))

        # Evaluate on unseen holdout test set
        test_eval = global_model.evaluate(X_test, y_test, verbose=0)
        test_loss = float(test_eval[0])
        test_acc  = float(test_eval[1])

        print(f"  --> Round {round_num} Global Model Updated | Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f}\n")

        history_records.append({
            "round": round_num,
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "test_loss": test_loss,
            "test_accuracy": test_acc
        })

    print(f"Custom FedAvg Training Complete for {task_name.upper()}.\n")
    return global_model, pd.DataFrame(history_records)


## 7. Execute Federated Training

We run 10 federated rounds for both Cardiovascular Disease (CVD) and Diabetes risk prediction models.


In [ ]:
# ---- Train CVD Global Model ----
cvd_global_model, cvd_history_df = run_federated_training(
    task_name="cvd",
    input_dim=len(cvd_feature_names),
    hospitals_data=cvd_hospitals,
    X_test=X_cvd_test,
    y_test=y_cvd_test,
    num_rounds=10,
    local_epochs=3,
    batch_size=32
)

# ---- Train Diabetes Global Model ----
diab_global_model, diab_history_df = run_federated_training(
    task_name="diabetes",
    input_dim=len(diab_feature_names),
    hospitals_data=diab_hospitals,
    X_test=X_diab_test,
    y_test=y_diab_test,
    num_rounds=10,
    local_epochs=3,
    batch_size=32
)


## 8. Global Model Evaluation on Unseen Holdout Test Set

The 20% holdout test partition was isolated **before any training began** and was never seen by any simulated hospital node during training.

We evaluate the aggregated global model on accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix. Genuine empirical values are reported without modification.


In [ ]:
def evaluate_global_model(model, X_test, y_test, task_name, feature_names):
    """
    Evaluates the aggregated global model on the held-out test cohort.
    Returns genuine metrics without modification.
    """
    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.50).astype(int)

    acc  = float(accuracy_score(y_test, y_pred))
    prec = float(precision_score(y_test, y_pred, zero_division=0))
    rec  = float(recall_score(y_test, y_pred, zero_division=0))
    f1   = float(f1_score(y_test, y_pred, zero_division=0))
    try:
        auc = float(roc_auc_score(y_test, y_prob))
    except Exception:
        auc = 0.50
    cm = confusion_matrix(y_test, y_pred)

    print("=" * 65)
    print(f"  GLOBAL MODEL EVALUATION -- {task_name.upper()}")
    print("=" * 65)
    print(f"  Test Cohort Size  : {len(y_test)} patients (20% holdout -- never seen during training)")
    print(f"  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision         : {prec:.4f}")
    print(f"  Recall            : {rec:.4f}")
    print(f"  F1-Score          : {f1:.4f}")
    print(f"  ROC-AUC           : {auc:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"    TN={cm[0,0]:4d}   FP={cm[0,1]:4d}")
    print(f"    FN={cm[1,0]:4d}   TP={cm[1,1]:4d}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['Low Risk (0)', 'High Risk (1)'])}")

    return {
        "model_name": f"{task_name}_federated_global_model",
        "training_mode": "Custom TensorFlow FedAvg",
        "evaluation_timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_test_samples": int(len(y_test)),
        "accuracy":  round(acc, 4),
        "precision": round(prec, 4),
        "recall":    round(rec, 4),
        "f1_score":  round(f1, 4),
        "roc_auc":   round(auc, 4),
        "confusion_matrix": {
            "true_negatives": int(cm[0, 0]),  "false_positives": int(cm[0, 1]),
            "false_negatives": int(cm[1, 0]), "true_positives":  int(cm[1, 1])
        },
        "feature_names": feature_names,
        "num_simulated_hospitals": 3,
        "num_federated_rounds": 10
    }, cm

# Evaluate CVD
cvd_metrics, cm_cvd = evaluate_global_model(cvd_global_model, X_cvd_test, y_cvd_test, "cvd", cvd_feature_names)

# Evaluate Diabetes
diab_metrics, cm_diab = evaluate_global_model(diab_global_model, X_diab_test, y_diab_test, "diabetes", diab_feature_names)


## 9. Federated Convergence Plots

Training loss and test accuracy trajectories across all federated rounds.


In [ ]:
OUTPUT_DIR = "federated_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def plot_convergence(history_df, task_name, output_dir=OUTPUT_DIR):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"MediSphere Federated Learning -- {task_name.upper()} Convergence [Custom FedAvg]",
                 fontsize=13, fontweight="bold")

    rounds = history_df["round"]
    losses = history_df["train_loss"]
    accs   = history_df["test_accuracy"] * 100

    ax1.plot(rounds, losses, marker='o', color="#1E88E5", linewidth=2.5, markersize=6,
             label="Weighted Train Loss")
    ax1.set_xlabel("Federated Round Number")
    ax1.set_ylabel("Binary Crossentropy Loss")
    ax1.set_title("Loss vs. Federated Round")
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend()

    ax2.plot(rounds, accs, marker='s', color="#43A047", linewidth=2.5, markersize=6,
             label="Test Accuracy")
    ax2.set_xlabel("Federated Round Number")
    ax2.set_ylabel("Accuracy (%)")
    ax2.set_title("Test Accuracy vs. Federated Round")
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend()

    plt.tight_layout()
    out_path = os.path.join(output_dir, f"{task_name}_loss_plot.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {out_path}")

plot_convergence(cvd_history_df,  "cvd")
plot_convergence(diab_history_df, "diabetes")


## 10. Model Export & Output Package

All trained artifacts are organized into `federated_outputs/` and compressed into a ZIP package:
```
federated_outputs/
├── cvd_global_model.keras          ← Keras native format
├── cvd_global_model/               ← SavedModel directory format
├── diabetes_global_model.keras
├── diabetes_global_model/
├── cvd_metrics.json
├── diabetes_metrics.json
├── cvd_training_history.csv
├── diabetes_training_history.csv
├── cvd_loss_plot.png
├── diabetes_loss_plot.png
└── README.txt                      ← Model card with provenance and disclaimer
```


In [ ]:
import shutil

# 1. Save Keras models (both .keras and SavedModel formats)
for task_name, model in [("cvd", cvd_global_model), ("diabetes", diab_global_model)]:
    keras_path = os.path.join(OUTPUT_DIR, f"{task_name}_global_model.keras")
    saved_path = os.path.join(OUTPUT_DIR, f"{task_name}_global_model")
    model.save(keras_path)
    model.save(saved_path)
    print(f"Saved {task_name} model: {keras_path}")
    print(f"Saved {task_name} model: {saved_path}/  (SavedModel dir)")

# 2. Save Metrics JSON
for task_name, metrics_dict in [("cvd", cvd_metrics), ("diabetes", diab_metrics)]:
    metrics_path = os.path.join(OUTPUT_DIR, f"{task_name}_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics_dict, f, indent=4)
    print(f"Saved {metrics_path}")

# 3. Save Training History CSVs
cvd_history_df.to_csv(os.path.join(OUTPUT_DIR, "cvd_training_history.csv"), index=False)
diab_history_df.to_csv(os.path.join(OUTPUT_DIR, "diabetes_training_history.csv"), index=False)
print("Saved training history CSVs")

# 4. Write README / Model Card
readme = f"""========================================================================
MEDISPHERE COGNITIVE TWIN -- FEDERATED LEARNING GLOBAL MODELS
========================================================================
Trained via Custom TensorFlow Federated Averaging across 3 SIMULATED Hospitals.

Training Engine:       Custom TensorFlow FedAvg (Native Keras / NumPy)
Date of Training:      {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Federated Rounds:      10
Local Epochs/Round:    3

SIMULATED HOSPITALS (educational partitions, NOT real institutions):
  Hospital 1: Regional Medical Center   -- 40% of training data
  Hospital 2: Urban University Hospital -- 35% of training data
  Hospital 3: Community Health Clinic   -- 25% of training data

MODELS:
1. cvd_global_model.keras
   Feature count : {len(cvd_feature_names)}
   Features      : {cvd_feature_names}
   Test Accuracy : {cvd_metrics['accuracy']:.4f}
   Test ROC-AUC  : {cvd_metrics['roc_auc']:.4f}
   Test F1-Score : {cvd_metrics['f1_score']:.4f}

2. diabetes_global_model.keras
   Feature count : {len(diab_feature_names)}
   Features      : {diab_feature_names}
   Test Accuracy : {diab_metrics['accuracy']:.4f}
   Test ROC-AUC  : {diab_metrics['roc_auc']:.4f}
   Test F1-Score : {diab_metrics['f1_score']:.4f}

HOW TO LOAD IN FLASK:
   from tensorflow import keras
   model = keras.models.load_model('cvd_global_model.keras')
   pred  = model.predict(scaled_features_array)

DISCLAIMER:
   This is an educational research prototype for the MediSphere Cognitive
   Twin project. It has NOT been evaluated, certified, or cleared by any
   medical regulatory authority (FDA, EMA, CDSCO). No real hospitals,
   patients, or clinical systems are connected. No encryption or
   Differential Privacy is implemented. Do not use for clinical decisions.
========================================================================
"""
with open(os.path.join(OUTPUT_DIR, "README.txt"), "w") as f:
    f.write(readme)
print("Saved README.txt")

# 5. Compress output folder into ZIP
zip_name = "federated_outputs"
shutil.make_archive(zip_name, "zip", OUTPUT_DIR)
print(f"\nCompressed outputs to {zip_name}.zip")


## 11. Educational Notes & Clinical Disclaimer

---
### 11.1 Custom TensorFlow FedAvg Mechanics
Federated Averaging (McMahan et al., 2017) trains models across decentralized clients without centralizing private data:
1. **Broadcast**: The server sends current global weights $W_{\text{global}}$ to each node.
2. **Local Training**: Each hospital updates weights locally for $E$ epochs on private data.
3. **Parameter Aggregation**: The server computes sample-weighted average:
   $$W_{\text{global}} = \sum_{k=1}^K \frac{n_k}{N} W_k$$

---
### 11.2 What Information is Shared vs. Kept Private?
| Data Category | Shared with Server? |
|:---|:---:|
| Patient age, sex, vitals, lab results, EHR records | ❌ No — stays inside hospital node |
| Neural network weights ($W$, $b$) | ✅ Yes — transmitted per round |
| Aggregated loss/accuracy scalars | ✅ Yes — for monitoring |

---
### 11.3 Clinical Disclaimer
> **MANDATORY NOTICE**: This software is an educational and architectural research prototype developed for the MediSphere Cognitive Twin academic project.
>
> - It has **not** been evaluated, certified, or approved by the U.S. FDA, European EMA, or any medical regulatory authority.
> - No real hospital systems, electronic health records, or active clinical workflows are connected.
> - Metrics are derived from retrospective public research datasets and do not constitute clinical validation.
> - This tool must **never** be used to make or override autonomous medical diagnoses or treatment decisions without a licensed physician.
